In [ ]:
import yaml
from omegaconf import OmegaConf

import torch
import einops
import matplotlib.pyplot as plt

from ts2.data.histology_data_module import PatchDataModule

In [ ]:
cf_str = """
infra:
  seed: 1000
data:
  set: he
  parser:
    which: CachedCSVParser
    params:
      cached_parser_file:
        train: /nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/tsmeta/tcga_glioma_usup/695a2029_slide_all
        trainval: /nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/tsmeta/tcga_glioma_usup/695a2029_slide_toy
  transform:
    train:
      base_aug_params: {}
        #laser_noise_config: null
        #base_aug: three_channels
      strong_aug_params:
        aug_list: 
        - which: random_horiz_flip
          params: {}
        - which: random_vert_flip
          params: {}
        - which: random_autocontrast
          params: {}
        - which: random_sharpness
          params:
            sharpness_factor: 2
        - which: color_jitter
          params:
            brightness: 0.4
            contrast: 0.4 
            saturation: 0.4
            hue: 0.2
        - which: gaussian_blur
          params:
            kernel_size: 29
            sigma: 1
        - which: gaussian_noise
          params: {}
        - which: random_resized_crop
          params:
            size: 300
            scale: [0.08, 1.0]
            ratio: [0.8, 1.25]
        #- which: random_solarize
        #  params:
        #    threshold: 0
        #- which: random_erasing
        #  params: {}
        #- which: random_affine
        #  params:
        #    degrees: 10
        #    translate: [0.1, 0.3]
        aug_prob: 0.5
    trainval: ${.train}
  dataset:
      which: InterPatchJEPADataset
      which_process_read: memmap
      params:
        common:
          data_root: /nfs/mm-isilon/brainscans/dropbox/data/root_histology_db/he.plip
          num_transforms: 2
          num_context_samples: 3
          num_target_samples: 4
          balance_instance_class: False
        train:
          num_instance_self_replicate: 30
        trainval: {}
  loader:
    params:
      common:
        pin_memory: False
        persistent_workers: True
        prefetch_factor: 1
        num_workers: 1
      train:
        batch_size: 512
        drop_last: True
        shuffle: True
      trainval:
        batch_size: 8
        drop_last: False
        shuffle: False
"""

In [ ]:
cf = OmegaConf.create(yaml.safe_load(cf_str))
pdm = PatchDataModule(cf) 
pdm.prepare_data() 
pdm.setup(stage="fit") 

In [ ]:
tl = pdm.train_dataloader()
for i in range(10):  #len(tl.dataset)):
    data = tl.dataset.__getitem__(i)

    fig, axs = plt.subplots(3, 5)
    for i, (cpi, ci) in enumerate(zip(data["context_path"], data["context_image"])):
        axs[i, 0].imshow(
            (einops.rearrange(ci, "c h w -> h w c") * 255).to(int))
        axs[i, 0].axis("off")
        axs[i, 0].set_title(cpi.split("@")[-1])

    for i, (cpi, ci) in enumerate(zip(data["target_path"], data["target_image"])):
        for j, (tpj, tj) in enumerate(zip(cpi, ci)):
            axs[i, j + 1].imshow(
                (einops.rearrange(tj, "c h w -> h w c") * 255).to(int))
            axs[i, j + 1].axis("off")
            axs[i, j + 1].set_title(tpj.split("@")[-1])
